# pending writes와 장애 복구

같은 단계의 한 노드가 실패해도 성공한 병렬 노드가 다시 실행되지 않는지 확인합니다.

## 실행 환경 준비

Python 3.12 커널에 이 절에서 사용할 패키지 버전을 고정합니다.

In [1]:
# 실습에 필요한 패키지를 같은 버전으로 설치합니다.
%pip install -q langgraph==1.2.11 langgraph-checkpoint-sqlite==3.0.3

Note: you may need to restart the kernel to use updated packages.


## 병렬 그래프 준비

`good`과 `flaky`를 같은 단계에 놓고 로컬 로그로 호출 횟수를 셉니다.

In [2]:
# good과 flaky의 호출 횟수를 셀 로컬 파일과 병렬 상태 형식을 준비합니다.
from pathlib import Path
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph

candidates = [Path.cwd(), *Path.cwd().parents]
lab = next((p / "labs/part6/ch23-persistence" for p in candidates if (p / "labs/part6/ch23-persistence").is_dir()), None)
if lab is None:
    lab = next(p for p in candidates if p.name == "ch23-persistence")
trace_path = lab / "data/pending-trace.txt"
allow_path = lab / "data/allow-flaky"
trace_path.parent.mkdir(parents=True, exist_ok=True)
trace_path.unlink(missing_ok=True)
allow_path.unlink(missing_ok=True)

class State(TypedDict):
    results: Annotated[list[str], lambda left, right: left + right]

## 병렬 노드와 실행 경로

두 노드는 호출 이름을 기록하고 `flaky`만 첫 실행에서 실패합니다.

In [3]:
# good은 성공 결과를 쓰고 flaky는 허용 파일이 생기기 전까지 실제 RuntimeError를 냅니다.
def record(name: str) -> None:
    with trace_path.open("a", encoding="utf-8") as stream:
        stream.write(name + "\n")

def good(state: State) -> dict:
    record("good")
    return {"results": ["good-result"]}

def flaky(state: State) -> dict:
    record("flaky")
    if not allow_path.exists():
        raise RuntimeError("temporary failure")
    return {"results": ["flaky-result"]}

builder = StateGraph(State)
builder.add_node("good", good)
builder.add_node("flaky", flaky)
builder.add_edge(START, "good")
builder.add_edge(START, "flaky")
builder.add_edge("good", END)
builder.add_edge("flaky", END)
graph = builder.compile(checkpointer=InMemorySaver())

## 첫 실행의 부분 성공

오류가 발생한 뒤 저장된 pending write와 호출 로그를 읽습니다.

In [4]:
# 같은 superstep의 첫 실행을 실패시키고 저장된 task와 호출 순서를 확인합니다.
config = {"configurable": {"thread_id": "pending-23"}}
try:
    graph.invoke({"results": []}, config)
except RuntimeError as exc:
    print(type(exc).__name__, str(exc))
before_lines = trace_path.read_text(encoding="utf-8").splitlines()
snapshot = graph.get_state(config)
print("calls before resume:", before_lines)
print("tasks:", [(task.name, bool(task.error), len(task.result or [])) for task in snapshot.tasks])

RuntimeError temporary failure
calls before resume: ['good', 'flaky']
tasks: [('good', False, 1), ('flaky', True, 0)]


## 실패 원인을 없애고 재개

입력을 다시 넣지 않고 `None`으로 재개합니다.

In [5]:
# 허용 파일을 만든 뒤 입력 없이 재개해 good은 1회, flaky는 2회인지 검사합니다.
allow_path.write_text("ok", encoding="utf-8")
resumed = graph.invoke(None, config)
after_lines = trace_path.read_text(encoding="utf-8").splitlines()
print("calls after resume:", after_lines)
print("results:", sorted(resumed["results"]))
assert after_lines.count("good") == 1
assert after_lines.count("flaky") == 2
assert sorted(resumed["results"]) == ["flaky-result", "good-result"]

calls after resume: ['good', 'flaky', 'flaky']
results: ['flaky-result', 'good-result']
